# Exploratory Data Analysis (EDA)
## Online Retail Dataset

This notebook performs comprehensive exploratory data analysis on the cleaned retail dataset to uncover patterns, trends, and insights for downstream statistical analysis.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load cleaned data
df = pd.read_csv('../data/processed/cleaned_data.csv')
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

## 1. Data Overview

In [ ]:
# Display basic information
print("=" * 80)
print("DATASET INFORMATION")
print("=" * 80)
print(f"\nTotal Records: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print(f"\nColumn Data Types:")
print(df.dtypes)
print(f"\nFirst few rows:")
print(df.head(10))
print(f"\nDataset Info:")
df.info()

In [ ]:
# Missing values analysis
print("\n" + "=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_values.values,
    'Missing_Percent': missing_percent.values
})
print(missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percent', ascending=False))
if missing_df['Missing_Count'].sum() == 0:
    print("✓ No missing values detected!")

In [ ]:
# Summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
print(df.describe())

## 2. Revenue Analysis

In [ ]:
# Revenue statistics
print("REVENUE STATISTICS")
print("=" * 80)
print(f"Total Revenue: £{df['Revenue'].sum():,.2f}")
print(f"Average Transaction Revenue: £{df['Revenue'].mean():,.2f}")
print(f"Median Revenue: £{df['Revenue'].median():,.2f}")
print(f"Std Dev: £{df['Revenue'].std():,.2f}")
print(f"Min Revenue: £{df['Revenue'].min():,.2f}")
print(f"Max Revenue: £{df['Revenue'].max():,.2f}")

# Revenue distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Revenue'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Revenue (£)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Revenue Distribution')
axes[0].grid(axis='y', alpha=0.3)

# Box plot for outliers
axes[1].boxplot(df['Revenue'], vert=True)
axes[1].set_ylabel('Revenue (£)')
axes[1].set_title('Revenue Box Plot (Outlier Detection)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Identify outliers using IQR
Q1 = df['Revenue'].quantile(0.25)
Q3 = df['Revenue'].quantile(0.75)
IQR = Q3 - Q1
outlier_threshold = 1.5 * IQR
outliers = df[(df['Revenue'] > Q3 + outlier_threshold) | (df['Revenue'] < Q1 - outlier_threshold)]
print(f"\nOutliers detected: {len(outliers)} records ({len(outliers)/len(df)*100:.2f}%)")

## 3. Product Analysis

In [ ]:
# Product metrics
print("PRODUCT ANALYSIS")
print("=" * 80)
print(f"Total Unique Products: {df['StockCode'].nunique()}")
print(f"Total Transactions: {len(df):,}")

# Top 10 products by revenue
top_products_revenue = df.groupby(['StockCode', 'Description']).agg({
    'Revenue': 'sum',
    'Quantity': 'sum',
    'InvoiceNo': 'count'
}).reset_index()
top_products_revenue.columns = ['StockCode', 'Description', 'Total_Revenue', 'Total_Quantity', 'Transactions']
top_products_revenue = top_products_revenue.sort_values('Total_Revenue', ascending=False).head(10)

print("\nTop 10 Products by Revenue:")
print(top_products_revenue.to_string(index=False))

# Visualize top products
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue
axes[0].barh(range(len(top_products_revenue)), top_products_revenue['Total_Revenue'], color='steelblue')
axes[0].set_yticks(range(len(top_products_revenue)))
axes[0].set_yticklabels([desc[:30] + '...' if len(desc) > 30 else desc for desc in top_products_revenue['Description']], fontsize=9)
axes[0].set_xlabel('Total Revenue (£)')
axes[0].set_title('Top 10 Products by Revenue')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Quantity
axes[1].barh(range(len(top_products_revenue)), top_products_revenue['Total_Quantity'], color='coral')
axes[1].set_yticks(range(len(top_products_revenue)))
axes[1].set_yticklabels([desc[:30] + '...' if len(desc) > 30 else desc for desc in top_products_revenue['Description']], fontsize=9)
axes[1].set_xlabel('Total Quantity Sold')
axes[1].set_title('Top 10 Products by Quantity')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Price distribution
print(f"\nPrice Statistics:")
print(f"Average Unit Price: £{df['UnitPrice'].mean():.2f}")
print(f"Median Unit Price: £{df['UnitPrice'].median():.2f}")
print(f"Price Range: £{df['UnitPrice'].min():.2f} - £{df['UnitPrice'].max():.2f}")

## 4. Customer Analysis

In [ ]:
# Customer metrics
print("CUSTOMER ANALYSIS")
print("=" * 80)
print(f"Total Unique Customers: {df['CustomerID'].nunique()}")

# Top 10 customers by revenue
top_customers_revenue = df.groupby('CustomerID').agg({
    'Revenue': 'sum',
    'InvoiceNo': 'nunique',
    'Quantity': 'sum'
}).sort_values('Revenue', ascending=False).head(10)

print("\nTop 10 Customers by Revenue:")
print(top_customers_revenue)

# Visualize Top Customers
plt.figure(figsize=(12, 6))
sns.barplot(x=top_customers_revenue.index.astype(str), y='Revenue', data=top_customers_revenue, palette='viridis')
plt.title('Top 10 Customers by Revenue')
plt.xlabel('Customer ID')
plt.ylabel('Total Revenue (£)')
plt.xticks(rotation=45)
plt.show()

# Customer distribution by Country (Top 10)
customer_country = df.groupby('Country')['CustomerID'].nunique().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 6))
customer_country.plot(kind='bar', color='skyblue')
plt.title('Top 10 Countries by Number of Unique Customers')
plt.ylabel('Number of Customers')
plt.xticks(rotation=45)
plt.show()

## 5. Temporal Analysis

In [ ]:
# Convert InvoiceDate to datetime if not already
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Feature Engineering for Temporal Analysis
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour

# Monthly Revenue Trend
monthly_revenue = df.groupby('YearMonth')['Revenue'].sum()
plt.figure(figsize=(12, 6))
monthly_revenue.plot(kind='line', marker='o', color='green')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue (£)')
plt.grid(True)
plt.show()

# Day of Week Analysis
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
revenue_by_day = df.groupby('DayOfWeek')['Revenue'].sum().reindex(day_order)

plt.figure(figsize=(10, 6))
sns.barplot(x=revenue_by_day.index, y=revenue_by_day.values, palette='magma')
plt.title('Revenue by Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Total Revenue (£)')
plt.show()

# Hourly Analysis
revenue_by_hour = df.groupby('Hour')['Revenue'].sum()
plt.figure(figsize=(10, 6))
sns.lineplot(x=revenue_by_hour.index, y=revenue_by_hour.values, marker='s', color='orange')
plt.title('Revenue by Hour of the Day')
plt.xlabel('Hour (24h format)')
plt.ylabel('Total Revenue (£)')
plt.xticks(range(0, 24))
plt.grid(True)
plt.show()

## 6. Geographical Analysis

In [ ]:
# Revenue by Country
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False)

# Top 10 Countries by Revenue (Excluding UK for better scale)
top_countries_no_uk = country_revenue.drop('United Kingdom', errors='ignore').head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_countries_no_uk.values, y=top_countries_no_uk.index, palette='coolwarm')
plt.title('Top 10 International Markets by Revenue (Excluding UK)')
plt.xlabel('Total Revenue (£)')
plt.show()

if 'United Kingdom' in country_revenue.index:
    uk_rev_pct = (country_revenue['United Kingdom'] / country_revenue.sum() * 100)
    print(f"Percentage of Revenue from UK: {uk_rev_pct:.2f}%")

## 7. Correlation Analysis

In [ ]:
# Correlation Matrix
plt.figure(figsize=(10, 8))
corr_matrix = df[['Quantity', 'UnitPrice', 'Revenue', 'InvoiceMonth', 'InvoiceYear']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0)
plt.title('Correlation Heatmap')
plt.show()

# Quantity vs UnitPrice Relationship
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df[df['Quantity'] < 1000], x='UnitPrice', y='Quantity', alpha=0.5)
plt.title('Quantity vs Unit Price (Clipped Quantity < 1000)')
plt.show()

## 8. Summary and Conclusions
- **Revenue Catalysts**: The total revenue is significantly driven by a few high-value customers and specific peak months like November.
- **Product Concentration**: A small subset of products contributes to a large portion of the revenue, suggesting a Pareto distribution.
- **Seasonal Trends**: Clear evidence of seasonality exists, with revenues surging in Q4, likely due to holiday shopping.
- **Market Dominance**: While the UK remains the primary market (over 80% of revenue), international growth in Europe (Germany, France, Netherlands) is a key takeaway.
- **Shopping Behavior**: Peak shopping hours are between 10 AM and 3 PM, and Thursday appears to be the busiest day of the week.